<a href="https://colab.research.google.com/github/aadityane93/Toxic_Comments_Sentiment_Analysis/blob/aaditya/2_Neural_Network_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [6]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

## Load Files

In [8]:
# Load files from current Colab directory
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
test_labels = pd.read_csv("test_labels.csv")

# Remove rows where comment_text is missing
train_df = train_df.dropna(subset=["comment_text"])
test_df = test_df.dropna(subset=["comment_text"])

# Make sure comment_text is string
train_df["comment_text"] = train_df["comment_text"].astype(str)
test_df["comment_text"] = test_df["comment_text"].astype(str)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Test labels shape:", test_labels.shape)

# These are the 6 labels we want to predict
label_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Count how many toxic labels each comment has
train_df["label_count"] = train_df[label_cols].sum(axis=1)

# any_toxic = 1 if a comment has at least one toxic label
train_df["any_toxic"] = (train_df["label_count"] > 0).astype(int)

# Show how many comments have more than one label
print("Comments with more than one label:", (train_df["label_count"] > 1).sum())

Train shape: (159571, 8)
Test shape: (153164, 2)
Test labels shape: (153164, 7)
Comments with more than one label: 9865


## 3. CLEAN TEXT


In [11]:
# This function makes text simpler
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Keep only letters, numbers, and spaces
    text = re.sub(r"[^a-z0-9 ]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Return cleaned text
    return text


# Clean training comments
train_df["clean_text"] = train_df["comment_text"].apply(clean_text)

# Clean test comments
test_df["clean_text"] = test_df["comment_text"].apply(clean_text)

## SPLIT TRAINING DATA

In [12]:
# Split train.csv into training data and validation data
# stratify keeps similar clean/toxic ratio in train and validation data
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["any_toxic"]
)

# Reset indexes
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

# Check split sizes
print("Training data:", train_data.shape)
print("Validation data:", val_data.shape)

# Check toxic percentage in both splits
print("Training toxic percent:", train_data["any_toxic"].mean() * 100)
print("Validation toxic percent:", val_data["any_toxic"].mean() * 100)

Training data: (127656, 11)
Validation data: (31915, 11)
Training toxic percent: 10.167951369305008
Validation toxic percent: 10.167632774557418


## Tokenizer

In [13]:
# This function splits a sentence into words
def tokenize(text):
    # Split text by spaces
    return text.split()

## Build Vocabulary

In [14]:
# Maximum number of words we keep
MAX_VOCAB_SIZE = 50000

# Counter stores how many times each word appears
counter = Counter()

# Loop through every training comment
for text in train_data["clean_text"]:
    # Split comment into words
    words = tokenize(text)

    # Count the words
    counter.update(words)

# Create vocabulary with two special tokens
vocab = {
    "<PAD>": 0,   # Used for padding short comments
    "<UNK>": 1    # Used for unknown words
}

# Add most common words to vocabulary
for word, count in counter.most_common(MAX_VOCAB_SIZE - 2):
    # Give each word a unique number
    vocab[word] = len(vocab)

# Print vocabulary size
print("Vocabulary size:", len(vocab))

Vocabulary size: 50000


## Convert text to numbers

In [15]:
# Every comment will become exactly 200 words long
MAX_LEN = 200

# This function converts one comment into numbers
def encode_text(text):
    # Split comment into words
    words = tokenize(text)

    # Convert each word to its vocabulary number
    ids = [vocab.get(word, vocab["<UNK>"]) for word in words]

    # If comment is shorter than 200 words, add padding
    if len(ids) < MAX_LEN:
        ids = ids + [vocab["<PAD>"]] * (MAX_LEN - len(ids))

    # If comment is longer than 200 words, cut it
    else:
        ids = ids[:MAX_LEN]

    # Return list of word numbers
    return ids

##

## Create Dataset class

In [16]:
# This class prepares data for PyTorch
class ToxicDataset(Dataset):

    # This runs when we create the dataset
    def __init__(self, dataframe):
        # Store cleaned comments
        self.texts = dataframe["clean_text"].values

        # Store labels
        self.labels = dataframe[label_cols].values.astype(np.float32)

    # This returns the number of rows
    def __len__(self):
        return len(self.texts)

    # This returns one comment and its label
    def __getitem__(self, idx):
        # Convert one comment into numbers
        x = encode_text(self.texts[idx])

        # Convert comment numbers into PyTorch tensor
        x = torch.tensor(x, dtype=torch.long)

        # Convert labels into PyTorch tensor
        y = torch.tensor(self.labels[idx], dtype=torch.float)

        # Return comment and label
        return x, y

## Create Data loaders

In [17]:
# Batch size means how many comments the model sees at once
BATCH_SIZE = 128

# Create training dataset
train_dataset = ToxicDataset(train_data)

# Create validation dataset
val_dataset = ToxicDataset(val_data)

# Create training loader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Create validation loader
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

## Simple Neural Network

In [18]:
class SimpleTextNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels):
        super().__init__()

        # Converts word IDs into word vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # First hidden layer
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)

        # Dropout helps reduce overfitting
        self.dropout = nn.Dropout(0.3)

        # Final output layer
        self.fc2 = nn.Linear(hidden_dim, num_labels)

    def forward(self, x):
        # Convert words to embeddings
        embedded = self.embedding(x)

        # Mask padding tokens
        mask = (x != 0).unsqueeze(-1)

        # Ignore padding embeddings
        embedded = embedded * mask

        # Sum real word embeddings
        summed = embedded.sum(dim=1)

        # Count real words
        counts = mask.sum(dim=1).clamp(min=1)

        # Average embeddings
        pooled = summed / counts

        # Hidden layer
        hidden = F.relu(self.fc1(pooled))

        # Apply dropout
        hidden = self.dropout(hidden)

        # Final raw outputs
        logits = self.fc2(hidden)

        return logits

## Setup Model

In [21]:
# Use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create model
model = SimpleTextNN(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_dim=128,
    num_labels=6
).to(device)

# Count positive examples for each label
positive_counts = train_data[label_cols].sum().values

# Count negative examples for each label
negative_counts = len(train_data) - positive_counts

# pos_weight gives more importance to rare labels like threat and identity_hate
pos_weight_values = np.sqrt(negative_counts / positive_counts)

# Convert weights to PyTorch tensor
pos_weight = torch.tensor(pos_weight_values, dtype=torch.float).to(device)

# Loss function for multi-label classification
# BCEWithLogitsLoss is correct because we have 6 independent labels
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer updates model weights
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Print device
print("Using device:", device)

# Print positive weights
for label, weight in zip(label_cols, pos_weight_values):
    print(label, "pos_weight:", round(weight, 2))

Using device: cpu
toxic pos_weight: 9.42
severe_toxic pos_weight: 97.96
obscene pos_weight: 17.81
threat pos_weight: 334.94
insult pos_weight: 19.26
identity_hate pos_weight: 110.39


## Training Function

In [23]:

# This function trains the model for one epoch
def train_one_epoch():
    # Put model in training mode
    model.train()

    # Store total loss
    total_loss = 0

    # Loop through batches
    for x, y in train_loader:
        # Move data to GPU/CPU
        x = x.to(device)
        y = y.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Get model outputs
        logits = model(x)

        # Calculate loss
        loss = criterion(logits, y)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Add batch loss
        total_loss += loss.item()

    # Return average loss
    return total_loss / len(train_loader)

In [29]:
from sklearn.metrics import f1_score

def get_probs_and_labels(loader):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    return all_probs, all_labels


def find_best_thresholds(val_probs, val_labels):
    best_thresholds = []

    for i, label in enumerate(label_cols):

        best_threshold = 0.5
        best_f1 = 0

        # Try thresholds from 0.05 to 0.95
        for threshold in np.arange(0.05, 0.96, 0.05):

            preds = (val_probs[:, i] >= threshold).astype(int)

            f1 = f1_score(
                val_labels[:, i],
                preds,
                zero_division=0
            )

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold

        best_thresholds.append(best_threshold)

        print(label, "best threshold:", round(best_threshold, 2), "best F1:", round(best_f1, 4))

    return np.array(best_thresholds)


# Get validation probabilities
val_probs, val_labels = get_probs_and_labels(val_loader)

# Find best threshold for each label
best_thresholds = find_best_thresholds(val_probs, val_labels)

print("Best thresholds:", best_thresholds)

toxic best threshold: 0.85 best F1: 0.7255
severe_toxic best threshold: 0.95 best F1: 0.4354
obscene best threshold: 0.9 best F1: 0.7161
threat best threshold: 0.95 best F1: 0.3836
insult best threshold: 0.9 best F1: 0.6926
identity_hate best threshold: 0.95 best F1: 0.306
Best thresholds: [0.85 0.95 0.9  0.95 0.9  0.95]


## Evaluation function

In [31]:
# This function checks model performance
def evaluate(loader):
    # Put model in evaluation mode
    model.eval()

    # Store predicted probabilities
    all_probs = []

    # Store true labels
    all_labels = []

    # We do not need gradients during evaluation
    with torch.no_grad():

        # Loop through batches
        for x, y in loader:
            # Move data to GPU/CPU
            x = x.to(device)
            y = y.to(device)

            # Get model outputs
            logits = model(x)

            # Convert raw outputs into probabilities
            probs = torch.sigmoid(logits)

            # Store predictions
            all_probs.append(probs.cpu().numpy())

            # Store true labels
            all_labels.append(y.cpu().numpy())

    # Combine all batches
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    # Convert probabilities to 0 or 1 using threshold 0.5
    preds = (all_probs >= best_thresholds).astype(int)

    # Calculate ROC-AUC for each label
    auc_scores = []

    # Loop through all 6 labels
    for i in range(len(label_cols)):
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
        auc_scores.append(auc)

    # Average AUC across all labels
    mean_auc = np.mean(auc_scores)

    # Micro F1 gives overall performance across all labels
    micro_f1 = f1_score(all_labels, preds, average="micro", zero_division=0)

    # Macro F1 treats all labels equally, including rare labels
    macro_f1 = f1_score(all_labels, preds, average="macro", zero_division=0)

    # Hamming accuracy checks label-wise correctness
    # This is better than simple accuracy, but still not the main metric
    hamming_accuracy = (preds == all_labels).mean()

    # Exact match accuracy means all 6 labels must be correct for a comment
    exact_match_accuracy = (preds == all_labels).all(axis=1).mean()

    # Store per-label results
    per_label_results = []

    # Calculate precision, recall, and F1 for each label
    for i, label in enumerate(label_cols):
        precision = precision_score(all_labels[:, i], preds[:, i], zero_division=0)
        recall = recall_score(all_labels[:, i], preds[:, i], zero_division=0)
        f1 = f1_score(all_labels[:, i], preds[:, i], zero_division=0)

        per_label_results.append({
            "label": label,
            "auc": auc_scores[i],
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

    # Convert per-label results to DataFrame
    per_label_df = pd.DataFrame(per_label_results)

    # Return all useful metrics
    return mean_auc, micro_f1, macro_f1, hamming_accuracy, exact_match_accuracy, per_label_df

## Train Model

In [35]:
# Number of times model sees full training data
EPOCHS = 10

# Best validation AUC starts very low
best_val_auc = 0

# File where best model will be saved
best_model_path = "best_simple_nn.pt"

# Loop through epochs
for epoch in range(EPOCHS):
    # Train model for one epoch
    train_loss = train_one_epoch()

    # Check validation performance
    val_auc, val_micro_f1, val_macro_f1, val_hamming_acc, val_exact_acc, val_per_label_df = evaluate(val_loader)

    # Save model if validation AUC improves
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), best_model_path)

    # Print results
    print("Epoch:", epoch + 1)
    print("Train loss:", train_loss)
    print("Validation AUC:", val_auc)
    print("Validation Micro F1:", val_micro_f1)
    print("Validation Macro F1:", val_macro_f1)
    print("Validation Hamming Accuracy:", val_hamming_acc)
    print("Validation Exact Match Accuracy:", val_exact_acc)
    print("-------------------------")

# Print best validation AUC
print("Best Validation AUC:", best_val_auc)

# Show per-label validation result from last epoch
val_per_label_df

Epoch: 1
Train loss: 0.19435561403915855
Validation AUC: 0.9699559900053378
Validation Micro F1: 0.6794146072069585
Validation Macro F1: 0.5582487506612489
Validation Hamming Accuracy: 0.9757480808397305
Validation Exact Match Accuracy: 0.9025223249255836
-------------------------
Epoch: 2
Train loss: 0.17326514373621146
Validation AUC: 0.969130128269296
Validation Micro F1: 0.6835615505143402
Validation Macro F1: 0.5575303196515461
Validation Hamming Accuracy: 0.9757428586349156
Validation Exact Match Accuracy: 0.9013629954566819
-------------------------
Epoch: 3
Train loss: 0.15788760829440696
Validation AUC: 0.9680101460986831
Validation Micro F1: 0.6831427015250545
Validation Macro F1: 0.5635323555150551
Validation Hamming Accuracy: 0.9756958587915818
Validation Exact Match Accuracy: 0.901770327432242
-------------------------
Epoch: 4
Train loss: 0.14434700362936767
Validation AUC: 0.9676862485057528
Validation Micro F1: 0.6850010012682731
Validation Macro F1: 0.5651178515062022


,label,auc,precision,recall,f1
0,toxic,0.939065,0.737229,0.724885,0.731005
1,severe_toxic,0.979037,0.340984,0.681967,0.454645
2,obscene,0.968298,0.731369,0.761733,0.746242
3,threat,0.977004,0.319767,0.561224,0.407407
4,insult,0.964417,0.659610,0.751269,0.702462
5,identity_hate,0.949860,0.301435,0.486486,0.372230


## Preparation of Real test data

In [36]:
# test_labels.csv has some rows with -1
# -1 means those rows should not be used for scoring
valid_test_labels = test_labels[
    (test_labels[label_cols] != -1).all(axis=1)
]

# Merge test comments with their real labels
test_labeled_df = test_df.merge(valid_test_labels, on="id")

# Clean test text again after merging
test_labeled_df["clean_text"] = test_labeled_df["comment_text"].apply(clean_text)

# Check test data size
print("Real labeled test data:", test_labeled_df.shape)

Real labeled test data: (63978, 9)


## Test model

In [37]:
# Create test dataset with real labels
test_dataset = ToxicDataset(test_labeled_df)

# Create test loader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Load the best saved model before testing
model.load_state_dict(torch.load(best_model_path, map_location=device))

# Evaluate on test data
test_auc, test_micro_f1, test_macro_f1, test_hamming_acc, test_exact_acc, test_per_label_df = evaluate(test_loader)

# Print final test results
print("Test AUC:", test_auc)
print("Test Micro F1:", test_micro_f1)
print("Test Macro F1:", test_macro_f1)
print("Test Hamming Accuracy:", test_hamming_acc)
print("Test Exact Match Accuracy:", test_exact_acc)

# Show per-label test result
test_per_label_df

Test AUC: 0.9587539595736816
Test Micro F1: 0.5626053074978938
Test Macro F1: 0.4607659619665574
Test Hamming Accuracy: 0.9567194973272062
Test Exact Match Accuracy: 0.8573259557973053


,label,auc,precision,recall,f1
0,toxic,0.937771,0.526256,0.765189,0.623620
1,severe_toxic,0.978143,0.148857,0.762943,0.249110
2,obscene,0.954150,0.522736,0.728800,0.608804
3,threat,0.978665,0.247689,0.635071,0.356383
4,insult,0.949213,0.462368,0.725999,0.564941
5,identity_hate,0.954582,0.257432,0.608146,0.361738
